# Stop times — data cleaning

**Input:** `stop_times.csv` in this folder.  
**Output:** one row per `route_stop_id` × calendar date; standardized `time`; `date`; `datetime`; **`key_for_join`** = `route_stop_id|YYYY-MM-DD` (do not merge on `route_stop_id` alone).

1. Schema and types  
2. Explode `occurrences`  
3. Standardize `date`, `time`, `datetime`  
4. Key for join  
5. Missing keys  
6. Logical checks / export

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

INPUT_PATH = Path("<INPUT_DATA_DIR>/stop_times.csv")
OUTPUT_PATH = Path("<INPUT_DATA_DIR>/stop_times_cleaned.csv")

## 1. Schema and data types

In [ ]:
raw = pd.read_csv(INPUT_PATH)
print("Shape:", raw.shape)
print(raw.dtypes)
raw.head(3)

In [ ]:
df = raw.copy()

df["stop_time_id"] = pd.to_numeric(df["stop_time_id"], errors="coerce").astype("Int64")
df["route_stop_id"] = pd.to_numeric(df["route_stop_id"], errors="coerce").astype("Int64")
df["wait_time"] = pd.to_numeric(df["wait_time"], errors="coerce")

def parse_occurrences(cell):
    if pd.isna(cell):
        return None
    s = str(cell).strip()
    if not s:
        return None
    try:
        out = json.loads(s.replace("'", '"'))
    except json.JSONDecodeError:
        m = re.findall(r'"(\d{4}-\d{2}-\d{2})"', s)
        return m if m else None
    return out if isinstance(out, list) else None

df["occurrences_list"] = df["occurrences"].map(parse_occurrences)

df["_time_raw"] = df["time"].copy()
_ts = pd.to_datetime(df["time"], format="%H:%M:%S", errors="coerce")
df["time"] = _ts.dt.strftime("%H:%M:%S").where(_ts.notna(), pd.NA)

print("stop_time_id / route_stop_id:", df["stop_time_id"].dtype, df["route_stop_id"].dtype)
print("wait_time:", df["wait_time"].dtype)
print("time (after parse):", df["time"].dtype)
print("occurrences_list example:", df.loc[0, "occurrences_list"])

## 2. Explode occurrences

One row per `route_stop_id` and calendar date.

In [ ]:
long = df.explode("occurrences_list", ignore_index=True)
long = long.rename(columns={"occurrences_list": "service_date_str"})
long.head(6)

## 3. Standardize `date`, `time`, `datetime`

In [ ]:
long["date"] = pd.to_datetime(long["service_date_str"], format="%Y-%m-%d", errors="coerce")

long["datetime"] = pd.to_datetime(
    long["date"].dt.strftime("%Y-%m-%d") + " " + long["time"].astype(str),
    errors="coerce",
)

long[["date", "time", "datetime"]].head(5)

## 4. Key_for_join

Grain: **`(route_stop_id, date)`**. Column name: **`key_for_join`**.

In [ ]:
KEY_COL = "key_for_join"
long[KEY_COL] = (
    long["route_stop_id"].astype(str)
    + "|"
    + long["date"].dt.strftime("%Y-%m-%d")
)
long.loc[long["route_stop_id"].isna() | long["date"].isna(), KEY_COL] = pd.NA

dup = long.duplicated(KEY_COL, keep=False)
if dup.any():
    print("Duplicate key_for_join — sample:")
    print(long.loc[dup, ["stop_time_id", "route_stop_id", "date", "time", KEY_COL]].head(15))

## 5. Missing values in key fields

In [ ]:
KEY_COL = "key_for_join"
row_ok = (
    long["route_stop_id"].notna()
    & long["date"].notna()
    & long["time"].notna()
    & long[KEY_COL].notna()
)

## 6. Logical validation and export

In [ ]:
KEY_COL = "key_for_join"

invalid_wait = long["wait_time"] < 0
invalid_time = long["time"].isna() & long["_time_raw"].notna()
invalid_date = long["date"].isna() & long["service_date_str"].notna()

drop_from_out = ["occurrences", "service_date_str", "_time_raw"]
clean = long[row_ok & ~invalid_wait].drop(
    columns=[c for c in drop_from_out if c in long.columns],
    errors="ignore",
)
clean.to_csv(OUTPUT_PATH, index=False)

n_row_ok_fail = int((~row_ok).sum())
nw, nt, nd = int(invalid_wait.sum()), int(invalid_time.sum()), int(invalid_date.sum())
print("Validation anomaly counts (exploded rows, not stored as columns):")
print("  row_ok failures (incomplete keys):", n_row_ok_fail)
print("  invalid_wait:", nw)
print("  invalid_time:", nt)
print("  invalid_date:", nd)
print("Saved:", OUTPUT_PATH, "| rows:", len(clean))
clean.head(4)